Imports, paths, helpers

In [1]:
# === Setup ===
import os, re, json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# lightweight sentiment library
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

FINAL = Path("data/final")
SENT_DIR = Path("data/sentiment")
FINAL.mkdir(parents=True, exist_ok=True)
SENT_DIR.mkdir(parents=True, exist_ok=True)

def to_text(x): 
    try: return str(x)
    except: return ""

def clean_text(s: str) -> str:
    s = to_text(s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


Load reviews safely

In [2]:
# === Load reviews (dynamic output from Arjun's pipeline) ===
rev_path = FINAL / "reviews.csv"
assert rev_path.exists(), f"Missing {rev_path}"

reviews = pd.read_csv(rev_path)
# Ensure expected columns exist
for c in ["review_id","place_id","text","rating","lang","publish_time_utc","name","lat","lng","category"]:
    if c not in reviews.columns:
        reviews[c] = None

# Clean & filter
reviews["text"] = reviews["text"].apply(clean_text)
reviews = reviews[reviews["text"] != ""].copy()

# Optional: keep English-ish
if "lang" in reviews.columns:
    reviews = reviews[reviews["lang"].fillna("").astype(str).str.startswith(("en","EN","En")) | (reviews["lang"].isna())]

print("Loaded reviews:", reviews.shape)
reviews.head(2)


,place_id,name,lat,lng,address,types,rating,user_ratings_total,phone,website,opening_hours_present,source_query,category
0,ChIJF2_wv4lHDW0R47WKR5aSdmE,Onslow,-36.847685,174.769957,"9 Princes Street, Auckland Central, Auckland 1...","establishment,food,point_of_interest,restaurant",4.8,1289.0,+64 9 930 9123,http://www.onslow.nz/,True,restaurant in Auckland,restaurant
1,ChIJKWPi4S5HDW0R-Ao9My5Mf48,Gilt Brasserie,-36.847655,174.767196,"2 Chancery Street, Chambers, Auckland 1010, Ne...","establishment,food,point_of_interest,restaurant",4.6,613.0,+64 9 300 3126,https://giltbrasserie.nz/,True,restaurant in Auckland,restaurant
2,ChIJjz8Dgf5HDW0RcKfgUZdV8fw,Amano,-36.844412,174.770464,"66 - 68, Tyler Street, Britomart Place, Auckla...","bakery,establishment,food,point_of_interest,re...",4.6,4358.0,+64 9 394 1416,https://savor.co.nz/amano,True,restaurant in Auckland,restaurant
3,ChIJdQU6o-RHDW0RBI93MBK5gcU,Hello Beasty,-36.843112,174.762463,"95-97 Customs Street West, Auckland Central, A...","establishment,food,point_of_interest,restaurant",4.7,1394.0,+64 21 554 496,http://hellobeasty.nz/,True,restaurant in Auckland,restaurant
4,ChIJKZLQLLRHDW0R_F9vtz1cjiQ,alma,-36.844086,174.769165,"130 Quay Street, Auckland Central, Auckland 10...","establishment,food,point_of_interest,restaurant",4.7,405.0,+64 9 242 1570,http://www.alma.nz/,True,restaurant in Auckland,restaurant


Compute sentiment (VADER with rating fallback)

In [ ]:
# === Sentiment scoring ===
def score_sentiment(row):
    txt = row.get("text","") or ""
    vs = analyzer.polarity_scores(txt)  # {'neg':..,'neu':..,'pos':..,'compound':..}
    comp = vs["compound"]  # -1..+1

    # Optional: blend a little rating signal if present
    r = row.get("rating", None)
    if pd.notna(r):
        try:
            r = float(r)  # 1..5
            # map rating into -0.5..+0.5 and blend lightly
            comp = 0.8*comp + 0.2*((r-3)/2)
        except:
            pass

    if comp >= 0.25:
        label = "positive"
    elif comp <= -0.25:
        label = "negative"
    else:
        label = "neutral"

    return pd.Series({
        "compound": round(float(comp),4),
        "sentiment_label": label
    })

sent = reviews.apply(score_sentiment, axis=1)
reviews_scored = pd.concat([reviews, sent], axis=1)

print("Scored:", reviews_scored.shape)
reviews_scored[["text","compound","sentiment_label"]].head(3)


Per-place aggregation (totals and percentages)

In [6]:
# === Aggregate per place (for map/summary) ===
def pct(pos, total): 
    return round(100.0 * pos / total, 2) if total else 0.0

grp = (reviews_scored
       .groupby(["place_id","name","lat","lng","category"], dropna=False, as_index=False)
       .agg(
            review_count=("review_id","nunique"),
            avg_rating=("rating","mean"),
            avg_compound=("compound","mean"),
            pos_count=("sentiment_label", lambda s: (s=="positive").sum()),
            neg_count=("sentiment_label", lambda s: (s=="negative").sum()),
            neu_count=("sentiment_label", lambda s: (s=="neutral").sum()),
       ))

grp["pct_positive"] = grp.apply(lambda r: pct(r["pos_count"], r["review_count"]), axis=1)
grp["pct_negative"] = grp.apply(lambda r: pct(r["neg_count"], r["review_count"]), axis=1)
grp["pct_neutral"]  = grp.apply(lambda r: pct(r["neu_count"], r["review_count"]), axis=1)

# overall label per place
def overall_label(row):
    if row["pct_positive"] >= 60: return "positive"
    if row["pct_negative"] >= 40: return "negative"
    return "neutral"

grp["overall_label"] = grp.apply(overall_label, axis=1)

print("Place-level summary:", grp.shape)
grp.head(3)


,place_id,review_count,text
0,ChIJ--vYZXQVDW0R7aXJN3A2Aq4,5,[We had a medium roast lamb and roast beef din...
1,ChIJ-00UHBxKDW0RT8IBRwA5KsM,5,[Excellent lovely Chinese food plus they have ...
2,ChIJ-080qABDDW0R2ud1Ol0zm9c,5,"[Safe, secure,easy launching, Beautiful and tr..."
3,ChIJ-0FfMP0_DW0R5OFDizOoHqI,5,[Service was great! Guy at the cashier and ser...
4,ChIJ-0FfMP0_DW0RFC1TAOQM1Fc,5,"[Staff and place is great, but noticed the chi..."


Save outputs (CSV + Excel) and update runs.json

In [9]:
# === Save ===
# 1) Row-level scored reviews (CSV for downstream use if needed)
sent_csv = FINAL / "sentiment.csv"
reviews_scored[[
    "review_id","place_id","name","lat","lng","category",
    "rating","publish_time_utc","text","compound","sentiment_label"
]].to_csv(sent_csv, index=False)

# 2) Place-level Excel (similar to your shops_sentiment.xlsx, easy to demo)
xlsx_path = SENT_DIR / "shops_sentiment.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    grp.to_excel(writer, sheet_name="places", index=False)
    reviews_scored.head(2000).to_excel(writer, sheet_name="sample_reviews", index=False)

print(f"Saved:\n - {sent_csv}\n - {xlsx_path}")

# 3) Touch runs.json with a note
runs_path = FINAL/"runs.json"
try:
    runs = json.load(open(runs_path, "r", encoding="utf-8"))
except:
    runs = {}
runs["sentiment_last_built"] = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
with open(runs_path, "w", encoding="utf-8") as f:
    json.dump(runs, f, indent=2)
print("Updated runs.json")


Places dataset-
Rows: 6501, Columns: 13
Reviews dataset-
Rows: 24805, Columns: 9
Grouped reviews dataset-
Rows: 5508, Columns: 3
Merged dataset-
Rows: 5508, Columns: 8
